In [ ]:
smoothing = 0

In [ ]:
import pymc as pm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sparcl.client import SparclClient

In [ ]:
from astropy import units as u
from astropy.table import Table
from astropy.modeling import models


In [ ]:
import specutils
from specutils import Spectrum1D, SpectralRegion
from specutils.fitting.continuum import fit_continuum
from specutils.manipulation import noise_region_uncertainty, extract_region, gaussian_smooth, box_smooth
from specutils.fitting import find_lines_threshold, estimate_line_parameters, fit_lines, fit_generic_continuum

In [ ]:
client = SparclClient()

In [ ]:
import desi_utils

In [ ]:
import pyneb

In [ ]:
dwarf_cat = Table.read("stellar_mass_emline_dwarfs.fits")

In [ ]:
spectra_cat = Table.read("spectra_good.fits")

In [ ]:
spectra_cat

# Restart here

In [ ]:
np.random.choice(spectra_cat["targetid"])

In [ ]:
target_id = 39632991529468259

In [ ]:
idx = np.where(dwarf_cat["targetid"] == target_id)[0][0]

In [ ]:
inc = ['specid', 'redshift', 'flux', 'wavelength', 'spectype', 'specprimary', 'survey', 'program', 'targetid', 'redshift_warning']
res = client.retrieve_by_specid(specid_list = [target_id], include = inc,
                                dataset_list = ['DESI-DR1'])


In [ ]:
records = res.records

# Select the primary spectrum
spec_primary = np.array([records[jj].specprimary for jj in range(len(records))])

primary_ii = np.where(spec_primary == True)[0][0]



In [ ]:
def plot_spectrum(spectrum, **kwargs):
    plt.plot(spectrum.spectral_axis, spectrum.flux, **kwargs)

### Initial plot

In [ ]:
meas = dwarf_cat[idx]
meas

In [ ]:
z = meas["z"]
z

In [ ]:
flux_unit = 1e-17 * u.erg / u.s / u.cm**2 / u.angstrom

In [ ]:
λ = records[primary_ii].wavelength / (1 + z)
galspec = Spectrum1D(flux=records[primary_ii].flux * flux_unit, spectral_axis=λ * u.angstrom)

In [ ]:
if smoothing == 0:
    smoothspec = galspec
else:
    smoothspec = box_smooth(galspec, smoothing)

In [ ]:


plt.figure(figsize = (20, 6))

# Plot the original spectrum in maroon color
plot_spectrum(galspec)
plot_spectrum(smoothspec)
plt.axhline(0)

plt.xlabel(r'$\lambda$ [$\AA$] (observed)')
plt.show()



In [ ]:
good_lines = desi_utils.retrieve_good_lines(meas)
good_lines

# Finding & fitting lines

In [ ]:
cont_fit = fit_generic_continuum(smoothspec)

In [ ]:
cont_fit

In [ ]:
λ = smoothspec.spectral_axis
plt.plot(smoothspec.spectral_axis, smoothspec.flux)
plt.plot(λ, cont_fit(λ))


In [ ]:
flat_spec = smoothspec - cont_fit(λ)

In [ ]:
plt.plot(λ, flat_spec.flux)
plt.axhline(0, color="C1")

## Comparing with catalogue fits

In [ ]:
def print_measured_lines(lines, meas=meas, z=None, flux_scale=None):
    if z is None:
        z = meas["z"]

    if flux_scale is None:
        flux_scale = meas["flux_scale"]
        
    print(f"{'line':16}{'flux':12}{'fluxerr':12}{'λ0':12}")
    for line in lines:
        line = desi_utils.to_elsm_format(line)
        μ, f, ferr = meas[f"{line}_center"], meas[f"{line}_flux"], meas[f"{line}_fluxerr"]
        f /= flux_scale / (1+z)
        ferr /= flux_scale / (1+z)
        print(f"{line:16}{f:12.2f}{ferr:12.2f}{μ:12.2f}")

## Specutils fits

In [ ]:
def contsub(spectrum, lambda1, lambda2, lambda3, lambda4):
    '''
    Calculate the continuum in regions (lambda1, lambda2) and (lambda3, lambda4)
    Subtract the continuum and return a new spectrum for just the range (lambda1, lambda4) 
    '''
    imin = np.where(spectrum.spectral_axis.value > lambda1)[0][0]
    imax = np.where(spectrum.spectral_axis.value < lambda4)[0][-1]
    indxs = np.arange(imin, imax)
    flux = spectrum.flux
    wave = spectrum.spectral_axis
    spec2 = Spectrum1D(flux=flux[indxs], spectral_axis=wave[indxs])
    region = [(lambda1 * u.Angstrom, lambda2 * u.Angstrom), (lambda3 * u.Angstrom, lambda4 * u.Angstrom)]
    fitted_continuum = fit_continuum(spec2, window=region)
    contfit = fitted_continuum(spec2.spectral_axis)
    new_spec = Spectrum1D(flux=spec2.flux-contfit , spectral_axis=spec2.spectral_axis)

    new_spec.uncertainty = noise_region_uncertainty(new_spec, SpectralRegion(region)).uncertainty

    return new_spec


In [ ]:
def fitline(spectrum, lambda1, lambda2): 
    '''
    Fit a single emission line that is in the range (lambda1, lambda2) 
    '''
    sub_region = SpectralRegion(lambda1*u.Angstrom, lambda2*u.Angstrom)
    sub_spectrum = extract_region(spectrum, sub_region)
    line  = estimate_line_parameters(sub_spectrum, models.Gaussian1D())
    return line

In [ ]:
def fit_all_lines(subspec, line_windows, **kwargs):
    line_ic = []
    for window in line_windows:
        line_fit = fitline(subspec, window[0], window[1])
        if line_fit.mean > window[1]*u.angstrom:
            line_fit.mean = window[1]*u.angstrom
        if line_fit.mean < window[0]*u.angstrom:
            line_fit.mean = window[0]*u.angstrom
            
        line_ic.append(models.Gaussian1D(amplitude=line_fit.amplitude, mean=line_fit.mean, stddev=line_fit.stddev))
        print(line_fit)

    
    full_fit = fit_lines(subspec, line_ic, window=line_windows*u.angstrom)

    return full_fit

In [ ]:
def plot_line_models(subspec, line_fits):
    xrange = subspec.spectral_axis[[0, -1]]
    plt.plot(subspec.spectral_axis, subspec.flux, lw=3, color="black")
    plt.fill_between(subspec.spectral_axis.value, subspec.flux.value - subspec.uncertainty.array, subspec.flux.value + subspec.uncertainty.array,
                    alpha = 0.2)
        
    x = np.linspace(xrange[0], xrange[1], 1000)

    for linefit_ob in line_fits:
        model = (linefit_ob(x))
        plt.plot(x, model, alpha=1, lw=2)
    


In [ ]:
def print_results(line_fits):
    for fit in line_fits:
        flux = fit.amplitude * fit.stddev * np.sqrt(2*np.pi)

        flux = flux * (1+z) / 1e-17
        μ = fit.mean.value
        print(f"{μ:6.1f}\t{flux:8.4f}")
        

In [ ]:
def plot_line_fit(subspec, models, line, width=25):
    λ0 = desi_utils.retrieve_wavelength(line) 
    model_idx = np.argmin([np.abs(model.mean.value - λ0) for model in models])

    model = models[model_idx]

    xrange = (λ0 - width, λ0 + width)
    plot_spectrum(subspec, lw=3, color="black")
    plt.fill_between(subspec.spectral_axis.value, subspec.flux.value - subspec.uncertainty.array, subspec.flux.value + subspec.uncertainty.array,
                    alpha = 0.2, color="black")


    xmodel = np.linspace(*xrange, 1000)
    ymodel = model(xmodel * u.angstrom)
    plt.plot(xmodel, ymodel, alpha=1, lw=2)


    # plot catalogue model
    model_idx = np.argmin([np.abs(model.mean.value - λ0) for model in models])

    model = models[model_idx]

    
    plt.xlim(*xrange)
    plt.title(line)


## MCMC fitting


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Sequence

import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
import arviz as az


In [ ]:
@dataclass
class LineFitResult:
    trace: az.InferenceData
    model: pm.Model
    wave: np.ndarray
    flux: np.ndarray
    flux_err: np.ndarray
    line_names: list
    line_centers: list
    fit_continuum: bool

    def summary(self, var_names=None):
        """Return an arviz summary table of the posterior."""
        if var_names is None:
            var_names = ["amp", "mu", "sigma"] + (
                ["c0", "c1"] if self.fit_continuum else []
            )
        return az.summary(self.trace, var_names=var_names)

    def best_fit_curve(self, wave=None, hdi=True):
        """
        Return the posterior-mean model spectrum (and optionally the 94% HDI)
        evaluated on `wave` (defaults to the fitted wavelength grid).
        """
        if wave is None:
            wave = self.wave
        posterior = self.trace.posterior
        if hasattr(posterior, "to_dataset"):
            posterior = posterior.to_dataset()
        post = posterior.stack(sample=("chain", "draw"))
        amp = post["amp"].values      # (n_lines, n_samples)
        mu = post["mu"].values
        sigma = post["sigma"].values
        n_lines, n_samples = amp.shape

        w = wave[:, None, None]                       # (n_wave,1,1)
        a = amp[None, :, :]                            # (1,n_lines,n_samples)
        m = mu[None, :, :]
        s = sigma[None, :, :]
        lines = a * np.exp(-0.5 * ((w - m) / s) ** 2)  # (n_wave,n_lines,n_samples)
        model_samples = lines.sum(axis=1)              # (n_wave, n_samples)

        if self.fit_continuum:
            c0 = post["c0"].values[None, :]
            c1 = post["c1"].values[None, :]
            wave0 = self.wave.mean()
            model_samples = model_samples + c0 + c1 * (w[:, 0, :] - wave0)

        mean = model_samples.mean(axis=1)
        if not hdi:
            return mean, None, None
        lo = np.percentile(model_samples, 3, axis=1)
        hi = np.percentile(model_samples, 97, axis=1)
        return mean, lo, hi

    def plot_fit(self, ax=None, show_components=False):
        if ax is None:
            fig, ax = plt.subplots(figsize=(9, 5))

        ax.errorbar(
            self.wave, self.flux, yerr=self.flux_err,
            fmt=".", color="k", ms=12, label="data", zorder=2,
        )

        mean, lo, hi = self.best_fit_curve()
        ax.plot(self.wave, mean, color="crimson", lw=2, label="total fit", zorder=3)
        ax.fill_between(self.wave, lo, hi, color="crimson", alpha=0.2, zorder=2)

        if show_components:
            posterior = self.trace.posterior
            if hasattr(posterior, "to_dataset"):
                posterior = posterior.to_dataset()
            post_mean = posterior.mean(dim=("chain", "draw"))
            amp = post_mean["amp"].values
            mu = post_mean["mu"].values
            sigma = post_mean["sigma"].values
            for i, name in enumerate(self.line_names):
                comp = amp[i] * np.exp(-0.5 * ((self.wave - mu[i]) / sigma[i]) ** 2)
                if self.fit_continuum:
                    c0 = float(post_mean["c0"].values)
                    c1 = float(post_mean["c1"].values)
                    wave0 = self.wave.mean()
                    comp = comp + c0 + c1 * (self.wave - wave0)
                ax.plot(self.wave, comp, "--", lw=1.3, label=name, zorder=1)

        ax.set_xlabel("Wavelength")
        ax.set_ylabel("Flux")
        ax.legend(fontsize=8)
        return ax



    def plot_line_fit(self, line, ax=None, wave_range=30):
        if ax is None:
            fig, ax = plt.subplots(figsize=(9, 5))

        line_id = np.where(np.array(self.line_names) == line)[0][0]
        line_center = self.line_centers[line_id]

        filt = self.wave > line_center - wave_range
        filt &= self.wave < line_center + wave_range
        
        ax.errorbar(
            self.wave[filt], self.flux[filt], yerr=self.flux_err[filt],
            fmt=".", color="k", ms=12, alpha=1, label="data", zorder=3,
        )

        mean, lo, hi = self.best_fit_curve()
        ax.plot(self.wave[filt], mean[filt], color="crimson", lw=2, label="total fit", zorder=1)
        ax.fill_between(self.wave[filt], lo[filt], hi[filt], color="crimson", alpha=0.2, zorder=2)

        posterior = self.trace.posterior
        if hasattr(posterior, "to_dataset"):
            posterior = posterior.to_dataset()
        post_mean = posterior.mean(dim=("chain", "draw"))
        amp = post_mean["amp"].values
        mu = post_mean["mu"].values
        sigma = post_mean["sigma"].values
        
        comp = amp[line_id] * np.exp(-0.5 * ((self.wave - mu[line_id]) / sigma[line_id]) ** 2)
        if self.fit_continuum:
            c0 = float(post_mean["c0"].values)
            c1 = float(post_mean["c1"].values)
            wave0 = self.wave.mean()
            comp = comp + c0 + c1 * (self.wave - wave0)
        ax.plot(self.wave[filt], comp[filt], "--", lw=1.3, label=line, zorder=2)

        ax.set_xlabel("Wavelength")
        ax.set_ylabel("Flux")
        ax.legend(fontsize=8)
        
        return ax


    
    def plot_corner(self, var_names=None):
        if var_names is None:
            var_names = ["amp", "mu", "sigma"]
        return az.plot_pair(
            self.trace, var_names=var_names, kind="kde",
            marginals=True, figsize=(9, 9),
        )

    def line_fluxes(self):
        """
        Integrated flux of each Gaussian line: A * sigma * sqrt(2*pi),
        with uncertainty, computed from the posterior samples.
        """
        post = self.trace.posterior
        if hasattr(post, "to_dataset"):
            post = post.to_dataset()
        # amp and sigma are separate RVs with independent xarray dims even
        # though they share a shape, so multiply on the underlying arrays
        # (chain, draw, line) rather than via xarray broadcasting.
        amp_vals = post["amp"].values
        sigma_vals = post["sigma"].values
        integrated_vals = amp_vals * sigma_vals * np.sqrt(2 * np.pi)

        integrated = post["amp"].copy(data=integrated_vals)
        integrated = integrated.rename("integrated_flux").rename(
            {"amp_dim_0": "line"}
        )
        integrated = integrated.assign_coords(line=self.line_names)
        summ = az.summary(integrated)
        return summ

In [ ]:
def fit_lines(
    wave: np.ndarray,
    flux: np.ndarray,
    flux_err: np.ndarray | float | None,
    line_names: Sequence[str] | None = None,
    amp_guess: Sequence[float] | None = None,
    sigma_guess: Sequence[float] | float | None = None,
    center_prior_width: float = 2.0,
    sigma_bounds: tuple[float, float] = (0.3, 15.0),
    fit_continuum: bool = True,
    draws: int = 2000,
    tune: int = 2000,
    chains: int = 4,
    cores: int = 1,
    target_accept: float = 0.9,
    random_seed: int = 42,
    pixel_integrate: bool = True,
) -> LineFitResult:
    """
    Fit `n = len(line_centers)` Gaussian emission lines simultaneously.

    Parameters
    ----------
    wave, flux : 1D arrays
        Continuum-subtracted spectrum (same units for wave as line_centers).
    flux_err : 1D array, scalar, or None
        Per-pixel flux uncertainty. If None, a single noise scale is fit
        from the data (HalfNormal prior).
    line_centers : list of float
        Initial guesses for each line's central wavelength.
    line_names : list of str, optional
        Labels for output (defaults to Line_0, Line_1, ...).
    amp_guess : list of float, optional
        Initial amplitude guesses (defaults to max(flux) near each line).
    sigma_guess : lis
    t of float or float, optional
        Initial guess for line widths (defaults to ~3x median pixel spacing).
    center_prior_width : float
        Std dev (in wavelength units) of the Normal prior on each mu,
        centered at the corresponding line_centers entry. Keep this tight
        enough to prevent lines from swapping identities when blended.
    sigma_bounds : (low, high)
        Bounds used to build a bounded HalfNormal-like prior on sigma so it
        can't collapse to 0 or blow up across the whole window.
    fit_continuum : bool
        If True, fit a small linear residual continuum (c0 + c1*(wave-wave0)).
        draws, tune, chains, target_accept : PyMC sampling controls.

    Returns
    -------
    LineFitResult
    """
    wave = np.asarray(wave, dtype=float)
    flux = np.asarray(flux, dtype=float)
    n_lines = len(line_names)
    line_centers = [desi_utils.get_wavelength(line) for line in line_names]

    dw = np.median(np.diff(np.sort(wave)))
    
    if sigma_guess is None:
        sigma_guess = np.full(n_lines, 3 * dw)
    elif np.isscalar(sigma_guess):
        sigma_guess = np.full(n_lines, sigma_guess)
    sigma_guess = np.asarray(sigma_guess, dtype=float)

    if amp_guess is None:
        amp_guess = []
        for c in line_centers:
            mask = np.abs(wave - c) < 5 * dw
            amp_guess.append(flux[mask].max() if mask.any() else flux.max())
    amp_guess = np.asarray(amp_guess, dtype=float)
    amp_guess = np.clip(amp_guess, 1e-6, None)

    known_err = flux_err is not None
    if known_err and np.isscalar(flux_err):
        flux_err = np.full_like(flux, float(flux_err))

    wave0 = wave.mean()
    sigma_lo, sigma_hi = sigma_bounds


    if pixel_integrate:
        # Pixel edges = midpoints between neighboring samples, with the
        # two end pixels extrapolated symmetrically. Works for non-uniform
        # (e.g. log-lambda) grids too, as long as wave is sorted.
        order = np.argsort(wave)
        if not np.array_equal(order, np.arange(len(wave))):
            raise ValueError("`wave` must be sorted ascending for pixel_integrate=True")
        mid = 0.5 * (wave[:-1] + wave[1:])
        first_edge = wave[0] - (mid[0] - wave[0])
        last_edge = wave[-1] + (wave[-1] - mid[-1])
        edges = np.concatenate(([first_edge], mid, [last_edge]))
        pix_lo = edges[:-1]
        pix_hi = edges[1:]
        pix_dw = pix_hi - pix_lo


    with pm.Model() as model:
        wave_data = pm.Data("wave_data", wave)

        amp = pm.Normal("amp", sigma=amp_guess * 3, shape=n_lines,
                             initval=amp_guess)
        mu = pm.Normal("mu", mu=line_centers, sigma=center_prior_width,
                        shape=n_lines, initval=line_centers)
        # Bounded width: HalfNormal truncated to [sigma_lo, sigma_hi]
        sigma_raw = pm.HalfNormal("sigma_raw", sigma=sigma_guess, shape=n_lines,
                                   initval=sigma_guess)
        sigma = pm.Deterministic(
            "sigma", pt.clip(sigma_raw, sigma_lo, sigma_hi)
        )

 
        if pixel_integrate:
            # Exact analytic pixel-averaged flux density: integrate the
            # Gaussian across each pixel's [lo, hi] wavelength edges using
            # the error function, then divide by pixel width. This conserves
            # line flux even when a line spans only a handful of pixels,
            # unlike evaluating the Gaussian at the pixel center.
            lo_data = pm.Data("pix_lo", pix_lo)
            hi_data = pm.Data("pix_hi", pix_hi)
            dw_data = pm.Data("pix_dw", pix_dw)
 
            sqrt2 = np.sqrt(2.0)
            z_hi = (hi_data[:, None] - mu[None, :]) / (sigma[None, :] * sqrt2)
            z_lo = (lo_data[:, None] - mu[None, :]) / (sigma[None, :] * sqrt2)
            # integral of A*exp(-0.5*((w-mu)/sigma)^2) dw over [lo,hi]
            pixel_integral = (
                amp[None, :] * sigma[None, :] * np.sqrt(np.pi / 2.0)
                * (pt.erf(z_hi) - pt.erf(z_lo))
            )
            lines = pixel_integral / dw_data[:, None]
        else:
            # Point sampling at pixel centers (fast, but biased/flux-leaky
            # when sigma is only ~1-2 pixels wide).
            lines = amp[None, :] * pt.exp(
                -0.5 * ((wave_data[:, None] - mu[None, :]) / sigma[None, :]) ** 2
            )
        model_flux = lines.sum(axis=1)

        if fit_continuum:
            c0 = pm.Normal("c0", mu=0.0, sigma=np.nanstd(flux))
            c1 = pm.Normal("c1", mu=0.0, sigma=np.nanstd(flux) / max(np.ptp(wave), 1))
            model_flux = model_flux + c0 + c1 * (wave_data - wave0)

        pm.Deterministic("model_flux", model_flux)

        if known_err:
            pm.Normal("obs", mu=model_flux, sigma=flux_err, observed=flux)
        else:
            noise = pm.HalfNormal("noise", sigma=np.nanstd(flux))
            pm.Normal("obs", mu=model_flux, sigma=noise, observed=flux)

        trace = pm.sample(
            draws=draws, tune=tune, chains=chains, cores=cores,
            target_accept=target_accept, random_seed=random_seed,
            progressbar=True,
        )

    return LineFitResult(
        trace=trace, model=model, wave=wave, flux=flux,
        flux_err=flux_err if known_err else np.full_like(flux, np.nan),
        line_names=list(line_names), fit_continuum=fit_continuum, line_centers=list(line_centers),
    )



## Halpha, [NII] 6548, 6583

In [ ]:
region = (6450, 6500, 6630, 6680)

In [ ]:
subspec = contsub(flat_spec, *region)

In [ ]:
line_names = ["n_ii_6548", "h_alpha", "n_ii_6583"] #[   
line_centers = [desi_utils.get_wavelength(line) for line in line_names]

In [ ]:
result = fit_lines(
    subspec.spectral_axis, subspec.flux * 1e17, subspec.uncertainty.array * 1e17,
    line_names=line_names,
    draws=1000, tune=1000, chains=2,   # quick demo settings
)

In [ ]:
result.plot_fit()

In [ ]:
for line in line_names:
    result.plot_line_fit(line, wave_range=10)
    

In [ ]:
result.summary()

In [ ]:
result.line_fluxes()

In [ ]:
meas["flux_scale"]

In [ ]:
print_measured_lines(line_names, flux_scale=1, z=0)

## Sulfur II 6716 6732

In [ ]:
region = (6600, 6650, 6750, 6800)

In [ ]:
subspec = contsub(flat_spec, *region)

In [ ]:
line_names = ["sii6716", "sii6731"]
line_centers = [desi_utils.retrieve_wavelength(line) for line in line_names]

In [ ]:
result = fit_lines(
    subspec.spectral_axis, subspec.flux * 1e17, subspec.uncertainty.array * 1e17,
    line_centers=line_centers,
    line_names=line_names,
    draws=1000, tune=1000, chains=2,   # quick demo settings
)

In [ ]:
result.plot_fit()

In [ ]:
for line in line_names:
    result.plot_line_fit(line, wave_range=10)
    

In [ ]:
result.line_fluxes()

In [ ]:
print_measured_lines(line_names)

## S III 6312, OI 6300

In [ ]:
region = (6200, 6250, 6350, 6400)
subspec = contsub(flat_spec, *region)

In [ ]:
line_names = ["siii6312", "oi6300",]
line_centers = [desi_utils.retrieve_wavelength(line) for line in line_names]

In [ ]:
result = fit_lines(
    subspec.spectral_axis, subspec.flux * 1e17, subspec.uncertainty.array * 1e17,
    line_centers=line_centers,
    line_names=line_names,
    draws=1000, tune=1000, chains=2,   # quick demo settings
)

In [ ]:
result.plot_fit()

In [ ]:
for line in line_names:
    result.plot_line_fit(line, wave_range=10)
    

In [ ]:
result.line_fluxes()

In [ ]:
print_measured_lines(line_names, flux_scale=1)

### Hbeta, O III 4959, 5007

In [ ]:
region = (4700, 4800, 5050, 5150)

In [ ]:
subspec = contsub(flat_spec, *region)

In [ ]:
line_names = ["hbeta", "oiii4959", "oiii5007"]
line_centers = [desi_utils.retrieve_wavelength(line) for line in line_names]

In [ ]:
result = fit_lines(
    subspec.spectral_axis, subspec.flux * 1e17, subspec.uncertainty.array * 1e17,
    line_centers=line_centers,
    line_names=line_names,
    draws=1000, tune=1000, chains=2,   # quick demo settings
)

In [ ]:
result.plot_fit()

In [ ]:
for line in line_names:
    result.plot_line_fit(line, wave_range=10)
    

In [ ]:
result.line_fluxes()

In [ ]:
print_measured_lines(line_names, flux_scale=1)

## Hgamma

In [ ]:
line_names = ["hgama"]

# OII 3726, 3729

In [ ]:
region = (3600, 3700, 3770, 3860)

In [ ]:
subspec = contsub(flat_spec, *region)

In [ ]:
line_centers = [3726, 3729]
line_names = ["oii3726", "oii3729", ]

In [ ]:
result = fit_lines(
    subspec.spectral_axis, subspec.flux * 1e17, None,
    line_centers=line_centers,
    line_names=line_names,
    draws=1000, tune=1000, chains=2,  
    fit_continuum=True,
)

In [ ]:
result.plot_fit()
plt.xlim(3700, 3750)

In [ ]:
result.line_fluxes()

In [ ]:
print_measured_lines(line_names)